# 01 — Exploración del Dataset GreenHub Farmer

**Objetivo:** Entender la estructura del dataset antes de diseñar el pipeline y el schema del Data Warehouse.

**Herramientas:** `pandas`, `duckdb`, `pyarrow` (sin Spark — este notebook corre en el entorno `uv` local)

**Dataset:** [GreenHub Farmer en Kaggle](https://www.kaggle.com/datasets/hmatalonga/greenhub-farmer/data)

## 0. Verificar credenciales de Kaggle

In [1]:
import os
from pathlib import Path

kaggle_json = Path.home() / ".kaggle" / "kaggle.json"

if kaggle_json.exists():
    print(f"✅ Kaggle credentials encontradas en: {kaggle_json}")
else:
    print("❌ NO se encontró kaggle.json")
    print("   → Ve a https://www.kaggle.com/settings → API → 'Create New Token'")
    print("   → Guarda el archivo descargado en: C:\\Users\\TU_USUARIO\\.kaggle\\kaggle.json")

✅ Kaggle credentials encontradas en: C:\Users\aYo\.kaggle\kaggle.json


## 1. Descargar el dataset desde Kaggle

In [7]:
import subprocess
from pathlib import Path

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

DATASET = "hmatalonga/greenhub-farmer"

# Solo descarga si no existe ya
if not any(RAW_DIR.iterdir()) if RAW_DIR.exists() else True:
    print("📥 Descargando dataset...")
    result = subprocess.run(
        ["kaggle", "datasets", "download", "-d", DATASET, "-p", str(RAW_DIR), "--unzip"],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("❌ Error:", result.stderr)
    else:
        print("✅ Dataset descargado")
else:
    print("✅ Dataset ya existe en data/raw/")

📥 Descargando dataset...


Exception in thread Thread-6 (_readerthread):
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in po

Dataset URL: https://www.kaggle.com/datasets/hmatalonga/greenhub-farmer
License(s): apache-2.0


✅ Dataset descargado


In [8]:
# Ver qué archivos trae el dataset

raw_files = list(RAW_DIR.rglob("*"))

print(f"Archivos encontrados: {len(raw_files)}\n")

for f in sorted(raw_files)[:30]:
    size_mb = f.stat().st_size / (1024**2) if f.is_file() else 0
    print(f"  {f.relative_to(RAW_DIR)}  →  {size_mb:.2f} MB")

Archivos encontrados: 73

  devices  →  0.00 MB
  devices\part-0000.parquet  →  26.49 MB
  samples  →  0.00 MB
  samples\part-0000.parquet  →  26.88 MB
  samples\part-0001.parquet  →  30.14 MB
  samples\part-0002.parquet  →  28.87 MB
  samples\part-0003.parquet  →  28.69 MB
  samples\part-0004.parquet  →  28.21 MB
  samples\part-0005.parquet  →  28.15 MB
  samples\part-0006.parquet  →  28.65 MB
  samples\part-0007.parquet  →  27.55 MB
  samples\part-0008.parquet  →  28.39 MB
  samples\part-0009.parquet  →  28.62 MB
  samples\part-0010.parquet  →  28.55 MB
  samples\part-0011.parquet  →  28.75 MB
  samples\part-0012.parquet  →  28.81 MB
  samples\part-0013.parquet  →  29.01 MB
  samples\part-0014.parquet  →  28.98 MB
  samples\part-0015.parquet  →  29.18 MB
  samples\part-0016.parquet  →  28.74 MB
  samples\part-0017.parquet  →  28.79 MB
  samples\part-0018.parquet  →  28.30 MB
  samples\part-0019.parquet  →  32.07 MB
  samples\part-0020.parquet  →  29.92 MB
  samples\part-0021.parquet 

In [ ]:
## 2. Explorar la estructura de cada archivo Parquet
import pandas as pd

pqt_files = list(RAW_DIR.rglob("*.parquet"))
print(f"Archivos Parquet encontrados: {len(pqt_files)}\n")

summaries = []
for f in sorted(pqt_files):
    df = pd.read_parquet(f, engine="pyarrow").head(5)
    size_mb = f.stat().st_size / (1024**2)
    summaries.append({
        "archivo": f.name,
        "tamaño_MB": round(size_mb, 2),
        "columnas": len(df.columns),
        "nombres_columnas": list(df.columns)
    })
    print(f"📄 {f.name} ({size_mb:.1f} MB) — {len(df.columns)} columnas")
    print(f"   Columnas: {list(df.columns)}\n")

Archivos Parquet encontrados: 71

📄 part-0000.parquet (26.5 MB) — 7 columnas
   Columnas: ['id', 'model', 'manufacturer', 'brand', 'os_version', 'is_root', 'created_at']

📄 part-0000.parquet (26.9 MB) — 38 columnas
   Columnas: ['id', 'device_id', 'timestamp', 'battery_state', 'battery_level', 'timezone', 'country_code', 'memory_active', 'memory_inactive', 'memory_free', 'memory_user', 'charger', 'health', 'voltage', 'temperature', 'usage', 'up_time', 'sleep_time', 'network_status', 'network_type', 'mobile_network_type', 'mobile_data_status', 'mobile_data_activity', 'wifi_status', 'wifi_signal_strength', 'wifi_link_speed', 'screen_on', 'screen_brightness', 'roaming_enabled', 'bluetooth_enabled', 'location_enabled', 'power_saver_enabled', 'nfc_enabled', 'developer_mode', 'free', 'total', 'free_system', 'total_system']

📄 part-0001.parquet (30.1 MB) — 38 columnas
   Columnas: ['id', 'device_id', 'timestamp', 'battery_state', 'battery_level', 'timezone', 'country_code', 'memory_active', '

In [ ]:
## 3. Inspección profunda del archivo principal
import duckdb

# Usamos DuckDB para inspeccionar todos los Parquet de golpe sin cargar en RAM
con = duckdb.connect()

for f in sorted(pqt_files):
    print(f"=== {f.name} ===")
    try:
        #read_parquet(...)	DuckDB lee el archivo directamente en memoria sin cargarlo en pandas
        #Convierte la ruta Windows C:\...\archivo.parquet a formato Unix C:/.../archivo.parquet porque DuckDB requiere barras /
        result = con.execute(f"""
            DESCRIBE SELECT * FROM read_parquet('{str(f).replace(chr(92), '/')}')
        """).df()
        print(result.to_string(index=False))
        row_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{str(f).replace(chr(92), '/')}')").fetchone()[0]
        print(f"  → Total filas: {row_count:,}")
    except Exception as e:
        print(f"  Error: {e}")
    print()

=== part-0000.parquet ===
 column_name column_type null  key default extra
          id      BIGINT  YES None    None  None
       model     VARCHAR  YES None    None  None
manufacturer     VARCHAR  YES None    None  None
       brand     VARCHAR  YES None    None  None
  os_version     VARCHAR  YES None    None  None
     is_root      BIGINT  YES None    None  None
  created_at   TIMESTAMP  YES None    None  None
  → Total filas: 2,153,319

=== part-0000.parquet ===
         column_name column_type null  key default extra
                  id      BIGINT  YES None    None  None
           device_id      BIGINT  YES None    None  None
           timestamp   TIMESTAMP  YES None    None  None
       battery_state     VARCHAR  YES None    None  None
       battery_level      DOUBLE  YES None    None  None
            timezone     VARCHAR  YES None    None  None
        country_code     VARCHAR  YES None    None  None
       memory_active      BIGINT  YES None    None  None
     memory_ina

In [14]:
## 4. Análisis de nulos y distribución de tipos

# Analizar el archivo más importante (Batteries o el principal)
# Ajusta el nombre del archivo según lo que encuentres en la celda anterior
main_file = sorted(pqt_files)[1]  # se ajustará luego

df_main = pd.read_parquet(main_file, engine="pyarrow").head(50_000)

print(f"📊 Archivo: {main_file.name}")
print(f"   Filas en muestra: {len(df_main):,}")
print(f"   Columnas: {len(df_main.columns)}\n")

print("--- Tipos de datos ---")
print(df_main.dtypes)
print()

print("--- % de valores nulos ---")
nulls = (df_main.isnull().sum() / len(df_main) * 100).round(2)
print(nulls[nulls > 0].sort_values(ascending=False) if nulls.any() else "Sin nulos ✅")

📊 Archivo: part-0000.parquet
   Filas en muestra: 50,000
   Columnas: 38

--- Tipos de datos ---
id                               int64
device_id                        int64
timestamp               datetime64[us]
battery_state                      str
battery_level                  float64
timezone                           str
country_code                       str
memory_active                    int64
memory_inactive                  int64
memory_free                      int64
memory_user                      int64
charger                            str
health                             str
voltage                        float64
temperature                    float64
usage                          float64
up_time                        float64
sleep_time                     float64
network_status                     str
network_type                       str
mobile_network_type                str
mobile_data_status                 str
mobile_data_activity               str
wifi_s

In [35]:
# Estadísticas descriptivas de columnas numéricas

#df_main.describe(include='all').T[['count', 'unique', 'top', 'mean', 'min', 'max']].head(30)
#df_main.iloc[0].info()
df_main.head(5)



,id,device_id,timestamp,battery_state,battery_level,timezone,country_code,memory_active,memory_inactive,memory_free,...,roaming_enabled,bluetooth_enabled,location_enabled,power_saver_enabled,nfc_enabled,developer_mode,free,total,free_system,total_system
0,43539144,169297,1970-01-03 19:27:22,Discharging,76.0,America/Mexico_City,mx,327400,331644,902120,...,0.0,0.0,0.0,1.0,0.0,0.0,205.0,3676.0,307.0,2415.0
1,43541529,169297,1970-01-03 19:27:22,Discharging,76.0,America/Mexico_City,mx,327400,331644,902120,...,0.0,0.0,0.0,1.0,0.0,0.0,205.0,3676.0,307.0,2415.0
2,43541530,169297,1970-01-03 15:40:47,Discharging,80.0,America/Mexico_City,mx,330300,337632,902120,...,0.0,0.0,0.0,0.0,0.0,0.0,190.0,3676.0,307.0,2415.0
3,43541531,169297,1970-01-03 15:34:55,Discharging,81.0,America/Mexico_City,mx,329764,337940,902120,...,0.0,0.0,0.0,0.0,0.0,0.0,191.0,3676.0,307.0,2415.0
4,43541532,169297,1970-01-03 15:17:33,Discharging,82.0,America/Mexico_City,mx,325248,341200,902120,...,0.0,0.0,0.0,0.0,0.0,0.0,194.0,3676.0,307.0,2415.0


## 5. Identificar columnas para Particionado y Clustering

Esta sección es clave para diseñar el schema del Data Warehouse.

In [36]:
# Buscar columnas temporales (candidatas a PARTITION BY)
print("🕐 Columnas candidatas para PARTITION BY (temporales):")

time_cols = [c for c in df_main.columns if any(k in c.lower() for k in ['time', 'date', 'timestamp', 'created', 'day', 'hour'])]

for c in time_cols:
    print(f"  - {c}: {df_main[c].dtype}  | ejemplo: {df_main[c].dropna().iloc[0] if not df_main[c].dropna().empty else 'N/A'}")

print()

# Buscar columnas categóricas (candidatas a CLUSTER BY / INDEX)
print("🏷️  Columnas candidatas para CLUSTER BY (baja cardinalidad):")
for c in df_main.columns:
    nunique = df_main[c].nunique()
    if 2 <= nunique <= 50:
        print(f"  - {c}: {nunique} valores únicos | {df_main[c].value_counts().head(3).to_dict()}")


🕐 Columnas candidatas para PARTITION BY (temporales):
  - timestamp: datetime64[us]  | ejemplo: 1970-01-03 19:27:22
  - timezone: str  | ejemplo: America/Mexico_City
  - up_time: float64  | ejemplo: 239241.0
  - sleep_time: float64  | ejemplo: 203226.0

🏷️  Columnas candidatas para CLUSTER BY (baja cardinalidad):
  - battery_state: 4 valores únicos | {'Discharging': 25228, 'Charging': 15151, 'Not charging': 9477}
  - charger: 3 valores únicos | {'unplugged': 34666, 'ac': 12899, 'usb': 2248}
  - health: 6 valores únicos | {'Good': 49418, 'Unknown': 351, 'Overheat': 40}
  - network_status: 12 valores únicos | {'disconnected': 46333, 'WIFI': 2083, 'lte': 949}
  - network_type: 6 valores únicos | {'unknown': 45885, 'WIFI': 2257, 'MOBILE': 1382}
  - mobile_network_type: 12 valores únicos | {'0': 36842, 'hspa': 6075, 'lte': 2686}
  - mobile_data_status: 4 valores únicos | {'disconnected': 48126, 'connected': 1669, 'connecting': 16}
  - mobile_data_activity: 4 valores únicos | {'none': 48716,

## 6. Convertir algunos archivos Parquet a un único CSV consolidado

Este paso simula lo que hará el DAG de Airflow automáticamente.
Lo ejecutamos aquí manualmente **solo una vez** para exploración.

In [39]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [40]:
from pathlib import Path

f = sorted(pqt_files)[0]
output_csv = Path("../data/processed") / (f.stem + ".csv")

#con.execute(f"""
#    COPY (SELECT * FROM read_parquet('{str(f).replace(chr(92), '/')}'), SAMPLE_SIZE,3)
#    TO '{str(output_csv).replace(chr(92), '/')}' (FORMAT CSV, HEADER TRUE)
#""")

for f in sorted(pqt_files)[:3]:
    output_csv = PROCESSED_DIR / (f.stem + ".csv")
    con.execute(f"""
        COPY (SELECT * FROM read_parquet('{str(f).replace(chr(92), '/')}'))
        TO '{str(output_csv).replace(chr(92), '/')}' (FORMAT CSV, HEADER TRUE)
""")
    
print(f"✅ Guardado: {output_csv}")

print(f"✅ Guardado: {output_csv}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Guardado: ..\data\processed\part-0001.csv
✅ Guardado: ..\data\processed\part-0001.csv


In [ ]:
#PROCESSED_DIR = Path("../data/processed")
#PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Ver cuántas filas totales tendría el dataset completo
#total_rows = 0
#for f in sorted(pqt_files):
#    try:
#        count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{str(f).replace(chr(92), '/')}')").fetchone()[0]
#        size_mb = f.stat().st_size / (1024**2)
#        print(f"  {f.name:<40} {count:>10,} filas   {size_mb:.1f} MB")
#        total_rows += count
#    except Exception as e:
#        print(f"  {f.name}: ERROR — {e}")

# print(f"\n{'='*60}")
# print(f"  TOTAL estimado: {total_rows:,} filas")

#- timestamp: datetime64[us]  | ejemplo: 1970-01-03 19:27:22

# - battery_state: 4 valores únicos | {'Discharging': 25228, 'Charging': 15151, 'Not charging': 9477}
#  - charger: 3 valores únicos | {'unplugged': 34666, 'ac': 12899, 'usb': 2248}

## 7. Conclusiones para el diseño del Schema

> **Completa esta sección después de ejecutar las celdas anteriores.**

| Decisión | Columna | Justificación |
|----------|---------|---------------|
| `PARTITION BY` | `timestamp` | columna temporal — reduce el escaneo por rango de fechas |
| `CLUSTER BY` / `INDEX` | `battery_state` | baja cardinalidad — acelera filtros frecuentes en el dashboard - 4 valores únicos — acelera filtros frecuentes (Charging/Discharging) |
| Tabla principal (fact) | `samples` | archivo con más filas / más columnas relevantes y contiene todas las métricas de sensores |
| Tabla secundaria (dim) | `devices` | Información estática del dispositivo — se une a samples por device_id  |

**Modelo estrella:**

devices (dim)
    └── device_id ──► samples (fact)
                          ├── timestamp       → PARTITION BY
                          ├── battery_state   → CLUSTER BY / INDEX
                          ├── charger
                          ├── battery_level
                          ├── cpu_usage
                          └── memory_*

**Próximo paso:** Con esta información diseñaremos el schema SQL y el DAG de Airflow.